In [1]:
import pandas as pd
import numpy as np

In [2]:
# =====================================================
# Load Dataset & Basic Preprocessing
# =====================================================
df = pd.read_csv('../Data/TCS_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')
df = df.sort_index()

df["Daily_Return"] = df["Close"].pct_change() * 100
df["Daily_Range"] = df["Close"] - df["Low"]

In [3]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# 1 = (UP)
# 0 = (DOWN)

In [4]:
df.drop("Open", axis=1, inplace=True)
df.drop("High", axis=1, inplace=True)
df.drop("Low", axis=1, inplace=True)

In [5]:
df['lag_1'] = df['Close'].shift(1)
df['lag_7'] = df['Close'].shift(7)
df['lag_30'] = df['Close'].shift(30)

In [6]:
df['rolling_7_mean'] = df['Close'].shift(1).rolling(7).mean()
df['rolling_7_std'] = df['Close'].shift(1).rolling(7).std()

df['rolling_14_mean'] = df['Close'].shift(1).rolling(14).mean()
df['rolling_14_std'] = df['Close'].shift(1).rolling(14).std()

df['rolling_30_mean'] = df['Close'].shift(1).rolling(30).mean()
df['rolling_30_std'] = df['Close'].shift(1).rolling(30).std()

In [7]:
df['day_of_month'] = df.index.day

df["day_of_week"] = df.index.dayofweek

df["month"] = df.index.month

df["quarter"] = df.index.quarter

In [8]:
df['expanding_sum'] = df['Close'].expanding().sum()
df['expanding_mean'] = df['Close'].expanding().mean()
df['expanding_std'] = df['Close'].expanding().std()

In [9]:
import ta

# RSI — Overbought/Oversold signal
df['RSI'] = ta.momentum.RSIIndicator(
    df['Close'], window=14).rsi()

# MACD — Trend signal
macd = ta.trend.MACD(df['Close'])
df['MACD']        = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Hist']   = macd.macd_diff()

# Bollinger Bands — Volatility signal
bb = ta.volatility.BollingerBands(df['Close'], window=20)
df['BB_Upper'] = bb.bollinger_hband()
df['BB_Lower'] = bb.bollinger_lband()
df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']
df['BB_Pct']   = bb.bollinger_pband()  # 0-1 range me price position

print(df[['RSI','MACD','MACD_Signal','BB_Pct']].tail(10))

                  RSI       MACD  MACD_Signal    BB_Pct
Date                                                   
2024-12-17  50.252698  69.427856    91.340887  0.394238
2024-12-18  47.087199  52.883636    83.649437  0.244138
2024-12-19  55.155186  54.781998    77.875949  0.620575
2024-12-20  57.212174  60.172088    74.335177  0.725301
2024-12-23  49.480319  49.388687    69.345879  0.356780
2024-12-24  51.735312  44.974585    64.471620  0.463861
2024-12-26  51.020785  39.664339    59.510164  0.420717
2024-12-27  51.548290  36.032273    54.814586  0.417066
2024-12-30  54.784475  38.825928    51.616854  0.575216
2024-12-31  59.487192  50.335533    51.360590  0.848696


In [10]:
print(f"Total features created: {len(df.columns)}") 
print("\nFeature columns:")
print(df.columns.tolist())

Total features created: 29

Feature columns:
['Close', 'Volume', 'Daily_Return', 'Daily_Range', 'Target', 'lag_1', 'lag_7', 'lag_30', 'rolling_7_mean', 'rolling_7_std', 'rolling_14_mean', 'rolling_14_std', 'rolling_30_mean', 'rolling_30_std', 'day_of_month', 'day_of_week', 'month', 'quarter', 'expanding_sum', 'expanding_mean', 'expanding_std', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Pct']


In [11]:
nan_count = df.isnull().sum()
print(nan_count[nan_count > 0])

Daily_Return        1
lag_1               1
lag_7               7
lag_30             30
rolling_7_mean      7
rolling_7_std       7
rolling_14_mean    14
rolling_14_std     14
rolling_30_mean    30
rolling_30_std     30
expanding_std       1
RSI                13
MACD               25
MACD_Signal        33
MACD_Hist          33
BB_Upper           19
BB_Lower           19
BB_Width           19
BB_Pct             19
dtype: int64


In [12]:
print(f"\nOriginal rows : {len(df)}")

df = df.dropna()
print(f"After dropna  : {len(df)}")


Original rows : 3850
After dropna  : 3817


In [13]:
print(df['Target'].value_counts())
print(f"UP %   : {df['Target'].mean()*100:.1f}%")
print(f"DOWN % : {(1-df['Target'].mean())*100:.1f}%")

Target
1    2025
0    1792
Name: count, dtype: int64
UP %   : 53.1%
DOWN % : 46.9%


In [14]:
df.head(5)

,Close,Volume,Daily_Return,Daily_Range,Target,lag_1,lag_7,lag_30,rolling_7_mean,rolling_7_std,...,expanding_mean,expanding_std,RSI,MACD,MACD_Signal,MACD_Hist,BB_Upper,BB_Lower,BB_Width,BB_Pct
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-18,407.46,2258828,1.033995,3.91,0,403.29,405.81,437.73,404.282857,3.285309,...,423.101471,16.633163,42.329992,-6.072182,-6.487775,0.415593,425.909512,396.354488,29.555024,0.375757
2010-02-19,402.08,5405614,-1.320375,4.91,1,407.46,403.06,436.70,404.518571,3.467292,...,422.500857,16.767551,37.807439,-6.092905,-6.408801,0.315896,423.663627,396.255373,27.408254,0.212514
2010-02-22,402.87,2902857,0.196478,2.43,0,402.08,401.83,445.61,404.378571,3.554684,...,421.955556,16.847036,38.840741,-5.976686,-6.322378,0.345692,422.105668,396.003332,26.102335,0.263067
2010-02-23,393.57,2787023,-2.308437,5.35,0,402.87,399.52,449.46,404.527143,3.450622,...,421.188378,17.254432,32.082966,-6.559402,-6.369783,-0.189619,420.836476,394.396524,26.439952,-0.031260
2010-02-24,387.48,2769715,-1.547374,2.82,1,393.57,408.18,447.13,403.677143,5.186051,...,420.301316,17.876535,28.576682,-7.427006,-6.581228,-0.845779,421.106666,391.082334,30.024333,-0.119980


In [15]:
corr = df.corr(numeric_only=True)
corr['Close'].sort_values(ascending=False)

Close              1.000000
lag_1              0.999740
rolling_7_mean     0.999326
rolling_14_mean    0.998737
BB_Upper           0.998413
lag_7              0.998344
rolling_30_mean    0.997486
BB_Lower           0.997269
lag_30             0.992752
expanding_std      0.980197
expanding_mean     0.979509
expanding_sum      0.967382
rolling_30_std     0.772994
BB_Width           0.758914
rolling_14_std     0.756911
rolling_7_std      0.748995
Daily_Range        0.672935
MACD_Signal        0.149155
MACD               0.141166
quarter            0.050940
month              0.049945
Volume             0.011730
MACD_Hist          0.001120
day_of_week       -0.000112
day_of_month      -0.000209
Daily_Return      -0.012908
Target            -0.043297
BB_Pct            -0.075627
RSI               -0.100499
Name: Close, dtype: float64

In [16]:
features = [
    'Close',
    'Target',

    # Technical Indicators (actual signals)
    'RSI',
    'MACD',
    'MACD_Signal',
    'MACD_Hist',
    'BB_Pct',     
    'BB_Width',    # Volatility measure

    # Volatility
    'rolling_7_std',
    'Daily_Range',

    # Returns
    'Daily_Return',

    # Volume
    'Volume',

    # Calendar
    'day_of_week',
    'month',
    'quarter',
]

In [17]:
cols_to_save = features

df_save = df[cols_to_save].copy()

# Save
df_save.to_csv('features-column.csv', index=True)  

print(f"Saved!")
print(f"Shape: {df_save.shape}")
print(df_save.head())

Saved!
Shape: (3817, 15)
             Close  Target        RSI      MACD  MACD_Signal  MACD_Hist  \
Date                                                                      
2010-02-18  407.46       0  42.329992 -6.072182    -6.487775   0.415593   
2010-02-19  402.08       1  37.807439 -6.092905    -6.408801   0.315896   
2010-02-22  402.87       0  38.840741 -5.976686    -6.322378   0.345692   
2010-02-23  393.57       0  32.082966 -6.559402    -6.369783  -0.189619   
2010-02-24  387.48       1  28.576682 -7.427006    -6.581228  -0.845779   

              BB_Pct   BB_Width  rolling_7_std  Daily_Range  Daily_Return  \
Date                                                                        
2010-02-18  0.375757  29.555024       3.285309         3.91      1.033995   
2010-02-19  0.212514  27.408254       3.467292         4.91     -1.320375   
2010-02-22  0.263067  26.102335       3.554684         2.43      0.196478   
2010-02-23 -0.031260  26.439952       3.450622         5.35     